# Composite components: a battery from a YAML definition

This notebook walks the maintainer team through the composite prototype for
[PyPSA issue #1789](https://github.com/PyPSA/PyPSA/issues/1789). A
*composite* is a reusable recipe of fundamental components (here: bus, two
links, store) plus an optional fragment of optimisation math. The math is
written in [math-spec](https://github.com/energy-models/math-spec) and layered
onto the linopy model through `Model.add_spec` from
[linopy PR #954](https://github.com/PyPSA/linopy/pull/954).

The design rests on three ideas:

1. **A definition is data.** A YAML file names the members, the exposed
   parameters and the math. PyPSA validates it once at registration.
2. **An instance is nothing but components.** `add` creates ordinary PyPSA
   components and tags them. There is no hidden state to keep in sync with
   `n.remove`, `n.copy` or file export.
3. **The math binds existing variables.** The fragment never creates PyPSA
   variables. It refers to `Link-p_nom` and friends, which PyPSA already
   built, and adds expressions and constraints per instance.

Requirements: `uv sync --group composites` (linopy `incremental-spec` branch
and math-spec) and linopy's v1 semantics.

In [ ]:
import linopy
import numpy as np
import pandas as pd

import pypsa

linopy.options["semantics"] = "v1"

## A small host network

One electricity bus with a sinusoidal load, wind that peaks when the load is
low and an expensive gas backup. Batteries are worth building here.

In [ ]:
n = pypsa.Network()
n.set_snapshots(pd.date_range("2030-01-01", periods=48, freq="h"))
hours = np.arange(48)
n.add("Bus", "elec")
n.add("Load", "load", bus="elec", p_set=100 + 50 * np.sin(hours / 24 * 2 * np.pi))
n.add(
    "Generator",
    "wind",
    bus="elec",
    p_nom=300,
    marginal_cost=1,
    p_max_pu=np.clip(np.cos(hours / 24 * 2 * np.pi), 0, None),
)
n.add("Generator", "gas", bus="elec", p_nom=200, marginal_cost=80)

## The definition file

Everything about the battery lives in one YAML document. In a project this is
a file passed to `register` by path; here it is inline so the notebook is
self-contained. The same text ships as `examples/composites/battery.yaml`.

In [ ]:
BATTERY_YAML = """
name: battery
description: Battery storage as a DC bus, a charger, a discharger and a store.

parameters:
  bus: null
  carrier: battery
  p_nom_extendable: true
  p_nom: 0.0
  max_hours: 4.0
  efficiency: 0.95
  capital_cost: 0.0
  marginal_cost: 0.0
  availability: 1.0

components:
  Bus:
    dc:
      carrier: $carrier
  Link:
    charger:
      bus0: $bus
      bus1: "@dc"
      efficiency: $efficiency
      p_nom: $p_nom
      p_nom_extendable: $p_nom_extendable
      capital_cost: $capital_cost
      carrier: $carrier
    discharger:
      bus0: "@dc"
      bus1: $bus
      efficiency: $efficiency
      p_nom: $p_nom
      p_nom_extendable: $p_nom_extendable
      marginal_cost: $marginal_cost
      carrier: $carrier
  Store:
    store:
      bus: "@dc"
      e_nom_extendable: $p_nom_extendable
      carrier: $carrier

math:
  variables:
    Link_p_nom:
      foreach: [name]
    Link_p:
      foreach: [snapshot, name]
  expressions:
    p: sum(Link_p, by=discharger) - sum(Link_p, by=charger)
    p_nom: sum(Link_p_nom, by=discharger)
  constraints:
    symmetric_power:
      foreach: [battery]
      expression: sum(Link_p_nom, by=charger) == sum(Link_p_nom, by=discharger)
    discharge_availability:
      foreach: [snapshot, battery]
      expression: sum(Link_p, by=discharger) <= availability * sum(Link_p_nom, by=discharger)
"""
print(BATTERY_YAML)

Reading the file top to bottom:

- `parameters` are the knobs a user may pass to `add`. A `null` default
  (`bus`) makes the parameter required.
- `components` maps component class to member name to attributes. Two
  prefixes resolve at `add` time: `$x` inserts the exposed parameter `x`,
  `@m` inserts the component name of member `m` of the same instance. Member
  components are named `<instance>-<member>`, so `@dc` in instance `bat1`
  becomes `bat1-dc`.
- `math` is a math-spec fragment. `variables` lists which PyPSA model
  variables the fragment binds. The name `Link_p_nom` maps to linopy's
  `Link-p_nom`; math-spec identifiers cannot contain a dash. `expressions`
  are outputs you can read back after solving, `constraints` are added to the
  model.

The instance axis is called after the definition, here `battery`. Each member
of the bound class becomes a *lookup* from component names to instance names,
so `sum(Link_p_nom, by=charger)` collapses the charger links onto the
`battery` axis. That is the math-spec way to say "the charger of every
battery": the grammar has no label subscripts like `p[name=bat1-charger]`.

## Register and add instances

`register` accepts a path, YAML text, a dict or a `CompositeDefinition` (a
pydantic model, so a definition can also be written as a Python class) and
runs the validation: known component classes, unique member names, resolvable
`$`/`@` references, and a math fragment that binds a single component class.
The registered composite is reachable as `n.composites.battery`. Instances
are added either through that handle or through the ordinary `n.add` with the
composite name in place of a component class. Both accept a single name or a
list of names, `suffix` and `overwrite`, exactly like `n.add` does for components.

In [ ]:
battery = n.composites.register(BATTERY_YAML)
battery.add("bat1", bus="elec", capital_cost=10, efficiency=0.9)
n.add("battery", "bat2", bus="elec", capital_cost=20)
n.composites

`add` validates the parameters (required present, no unknown ones), fills the
defaults, resolves the references and calls `n.add` once per member. There is
nothing else. Try a bad call:

In [ ]:
try:
    battery.add("bat3", bus="elec", max_hour=2)
except ValueError as e:
    print(e)

## Instances are just tagged components

Every member carries two static columns, `composite` (the instance) and
`composite_type` (the definition). The `members` table is derived from those
columns on every access.

In [ ]:
battery.members

In [ ]:
n.c.links.static[
    ["bus0", "bus1", "efficiency", "capital_cost", "composite", "composite_type"]
]

Parameter values are stored under `composite_param_<name>` on the *primary
member*, the first member of the bound class (here `charger`): scalars as a
static column, time series as a dynamic attribute of the same name. They feed
the math layer later, so they must survive together with the network. Because
everything is a column, `n.copy()`, `n.export_to_netcdf()` and `n.remove` keep
instances intact. Only the definition itself is in memory and must be
registered again on a fresh network.

In [ ]:
n.c.links.static.filter(like="composite_param_")

In [ ]:
n.links

## From fragment to math-spec

`add_spec` needs a complete spec. `CompositeDefinition.spec()` expands the
fragment: it adds the `snapshot`, `name` and `battery` dimensions, one lookup
per member of the bound class, and one parameter for every exposed parameter
that the math bodies mention. A parameter is declared over `[battery]`, or over
`[snapshot, battery]` when `add_spec` finds a time series for it on any
instance; the shape follows the data, not the definition. Unused parameters are
left out on purpose: math-spec has one flat namespace, so declaring `p_nom` as
a parameter would collide with the `p_nom` output expression.

In [ ]:
print(battery.definition.spec_text())

The data behind that spec comes from `sources`, the mapping `add_spec` reads
names from. Building the model first, because the bound variables must exist.

In [ ]:
m = n.optimize.create_model()
sources = battery.sources(m)
{k: type(v).__name__ for k, v in sources.items()}

In [ ]:
sources["charger"]

Two things worth noting here. `name` is the full active `Link` index; a bound
variable may span a subset of a dimension, never a superset, and linopy
reindexes the variable onto it. And the lookup series is indexed by `name` and
named after the member, exactly as the spec declared it.

`create_model` calls `n.composites._add_spec_layers(model)` after the objective
is defined, so the layer is part of every model PyPSA builds. The layer name
is `composite-<definition>`.

In [ ]:
m.spec["composite-battery"]

The constraint has one row per instance and reads as intended:

In [ ]:
m.constraints["symmetric_power"]

## Solve and read back

In [ ]:
n.optimize.solve_model()
n.optimize.assign_solution()

In [ ]:
n.c.links.static[["p_nom_opt", "composite"]]

Output expressions carry a solution after solving. `expressions` converts them
to pandas with `snapshot` as the first axis. `p` is the net injection of the
battery into the host bus: discharge minus charge, so charging is negative.

In [ ]:
battery.expressions["p_nom"]

In [ ]:
battery.expressions["p"].plot()

## Statistics

The module registers a `composite` grouper, so any statistics call can
aggregate members per instance. Components outside a composite return NA and
drop out of the grouping.

In [ ]:
n.statistics.optimal_capacity(groupby="composite")

In [ ]:
n.statistics.energy_balance(groupby=["composite", "carrier"])

## Remove an instance

`remove` drops every member. Because the layer is rebuilt from the tagged
components on each `create_model`, the next optimisation sees one battery.

In [ ]:
battery.remove("bat2")
n.optimize()
battery.expressions["p_nom"]

## Time-varying parameters

An exposed parameter may be a time series. `add` stores it as a dynamic
attribute `composite_param_<name>` on the primary member, and `add_spec`
declares it over `[snapshot, battery]`, so the fragment sees the series. Here
`availability` caps the discharger through the `discharge_availability`
constraint: a battery that is under maintenance during the second morning.

In [ ]:
availability = pd.Series(1.0, index=n.snapshots)
availability["2030-01-02 06:00":"2030-01-02 12:00"] = 0.0
battery.add("bat3", bus="elec", capital_cost=10, availability=availability)
n.c.links.dynamic.composite_param_availability

`sources` hands the spec one frame per time-varying parameter, instances as
columns. Instances that were added with a scalar are broadcast over snapshots.

In [ ]:
n.optimize()
battery.sources(n.model)["availability"]

In [ ]:
n.c.links.dynamic.p[["bat1-discharger", "bat3-discharger"]].plot()

## Where the code lives

- `pypsa/composites/definition.py`: `CompositeDefinition`, validation,
  reference resolution and the fragment-to-spec expansion.
- `pypsa/composites/accessor.py`: `Composite` (`add`, `remove`, `members`,
  `sources`, `add_spec`, `expressions`), `CompositesAccessor`
  (`n.composites`) and the statistics grouper.
- `pypsa/networks.py`: the accessor is attached in `Network.__init__`.
- `pypsa/optimization/optimize.py`: `create_model` adds the layers after
  `define_objective`.
- `test/test_composites.py`: behaviour tests, skipped without math-spec.

## Known limits of the prototype

1. **One bound class per fragment.** linopy names every component axis
   `name`, and one dimension name means one axis in a spec. A constraint
   coupling `Store-e_nom` to `Link-p_nom` (the energy-to-power ratio) is
   therefore not expressible yet. Validation rejects such fragments with an
   explanatory error. Unlocking this needs a dimension alias when binding a
   variable in linopy.
2. **Extendable members only.** `Link-p_nom` exists for extendable links only,
   so members bound by the math should share the extendable flag.
3. **Definitions are in memory.** Instances persist as columns, definitions
   do not. A network loaded from file needs `register` again before
   optimising.
4. **Single investment period** for time-varying parameters. linopy's wide
   frame reader does not accept a `MultiIndex` snapshot axis yet, so a time
   series on a network with investment periods raises `NotImplementedError`.